# Spotify Tracks Dataset - Data Cleaning and Preprocessing

This notebook loads the raw Spotify tracks dataset, keeps the fields used by the dashboard, cleans invalid values, creates `Year` and `Popularity_Category`, and exports `dataset_5a/spotify_cleaned.csv`.


## 1. Import Libraries


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


## 2. Locate Input and Output Paths


In [2]:
PROJECT_DIR = Path.cwd()
DASH_DIR = PROJECT_DIR if PROJECT_DIR.name == "spotify_trends_dashboard" else PROJECT_DIR / "spotify_trends_dashboard"
if not DASH_DIR.exists():
    DASH_DIR = PROJECT_DIR

DATA_DIR = DASH_DIR / "dataset_5a"
DATA_DIR.mkdir(exist_ok=True)

candidates = [
    PROJECT_DIR / "dataset.csv",
    PROJECT_DIR / "spotify_tracks.csv",
    PROJECT_DIR / "spotify_tracks_dataset.csv",
    PROJECT_DIR / "spotify.csv",
    PROJECT_DIR.parent / "dataset.csv",
    DASH_DIR / "dataset.csv",
]

raw_path = next((p for p in candidates if p.exists()), None)
if raw_path is None:
    raise FileNotFoundError("No dataset found. Put dataset.csv in the project root or dashboard folder.")

print(f"Using file: {raw_path}")


Using file: /home/ahmed/Downloads/data-visualization-SpotifyTrends-main/dataset.csv


## 3. Load Raw Dataset


In [3]:
raw_df = pd.read_csv(raw_path, low_memory=False)
raw_df.columns = [c.strip() for c in raw_df.columns]

print("Raw shape:", raw_df.shape)
print("Columns:", list(raw_df.columns))


Raw shape: (114000, 21)
Columns: ['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']


## 4. Standardize Columns


In [4]:
def first_existing_column(df, options):
    lookup = {c.lower(): c for c in df.columns}
    for opt in options:
        if opt.lower() in lookup:
            return lookup[opt.lower()]
    return None


def coerce_year_int(value):
    num = pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]
    if pd.isna(num):
        return np.nan
    year = int(float(num))
    return year if 1900 <= year <= 2035 else np.nan


def extract_year_from_release(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if not text:
        return np.nan
    parsed = pd.to_datetime(text, errors="coerce")
    if pd.notna(parsed):
        return float(parsed.year)
    if len(text) >= 4 and text[:4].isdigit():
        return float(text[:4])
    return np.nan


column_aliases = {
    "track_id": ["track_id", "id"],
    "track_name": ["track_name", "title", "name", "track"],
    "artists": ["artists", "artist", "artist_name"],
    "track_genre": ["track_genre", "genre", "Genre"],
    "popularity": ["popularity", "track_popularity", "Popularity"],
    "duration_ms": ["duration_ms", "duration", "Duration"],
    "danceability": ["danceability"],
    "energy": ["energy"],
    "valence": ["valence"],
    "tempo": ["tempo"],
    "loudness": ["loudness"],
    "acousticness": ["acousticness"],
    "release_date": ["release_date", "album_release_date", "date"],
    "year": ["year", "Year"],
}

df = pd.DataFrame()
for target, options in column_aliases.items():
    src = first_existing_column(raw_df, options)
    if src is not None:
        df[target] = raw_df[src]

print("Mapped shape:", df.shape)
print("Mapped columns:", list(df.columns))


Mapped shape: (114000, 12)
Mapped columns: ['track_id', 'track_name', 'artists', 'track_genre', 'popularity', 'duration_ms', 'danceability', 'energy', 'valence', 'tempo', 'loudness', 'acousticness']


## 5. Clean, Validate, and Engineer Features

Convert numeric fields, load real release years from `dataset_5a/spotify_api_track_years.csv`, remove rows without a real year, validate feature ranges, create popularity categories, and keep the final dashboard columns. Run `spotify_api_year_enrichment.ipynb` first if the API year file does not exist.


In [ ]:
numeric_cols = [
    "popularity", "duration_ms", "danceability", "energy", "valence",
    "tempo", "loudness", "acousticness", "year",
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if "release_date" in df.columns:
    df["Year"] = df["release_date"].map(extract_year_from_release)
elif "year" in df.columns:
    df["Year"] = pd.to_numeric(df["year"], errors="coerce")
else:
    df["Year"] = np.nan

api_years_file = DATA_DIR / "spotify_api_track_years.csv"
if "track_id" in df.columns and api_years_file.exists():
    api_years_df = pd.read_csv(api_years_file)
    if {"track_id", "Year"}.issubset(api_years_df.columns):
        api_years_df = api_years_df[["track_id", "Year"]].dropna().copy()
        api_years_df["track_id"] = api_years_df["track_id"].astype(str)
        api_years_df["Year"] = api_years_df["Year"].map(coerce_year_int)
        api_years_df = api_years_df.dropna(subset=["Year"]).drop_duplicates("track_id", keep="last")
        year_map = api_years_df.set_index("track_id")["Year"]
        missing = df["Year"].isna()
        df.loc[missing, "Year"] = df.loc[missing, "track_id"].astype(str).map(year_map)
        print(f"Spotify API year matches: {df['Year'].notna().sum():,}")

if df["Year"].notna().sum() == 0:
    raise FileNotFoundError("No real Year source found. Run spotify_api_year_enrichment.ipynb to create dataset_5a/spotify_api_track_years.csv.")

df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df = df[df["Year"].notna()].copy()
df["Year"] = df["Year"].round().astype(int)
df = df[df["Year"].between(1900, 2035)]

required = [
    "track_name", "artists", "track_genre", "popularity", "danceability", "energy",
    "valence", "tempo", "loudness", "acousticness", "duration_ms",
]
df = df.dropna(subset=[c for c in required if c in df.columns]).copy()

for col in ["track_name", "artists", "track_genre"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

fake_genres = {"music", "unknown", "other", "n/a", "na", "nan", "none", ""}
if "track_genre" in df.columns:
    df = df[~df["track_genre"].str.lower().isin(fake_genres)].copy()

df["popularity"] = df["popularity"].clip(0, 100)
for col in ["danceability", "energy", "valence", "acousticness"]:
    if col in df.columns:
        df[col] = df[col].clip(0, 1)

if "duration_ms" in df.columns:
    tiny_duration = df["duration_ms"] < 2000
    df.loc[tiny_duration, "duration_ms"] = df.loc[tiny_duration, "duration_ms"] * 1000
    df = df[df["duration_ms"].between(30000, 900000)].copy()

df["Popularity_Category"] = pd.cut(
    df["popularity"],
    bins=[-1, 29, 69, 100],
    labels=["Low", "Medium", "High"],
)
df = df.dropna(subset=["Popularity_Category"]).copy()
df["Popularity_Category"] = pd.Categorical(
    df["Popularity_Category"],
    categories=["Low", "Medium", "High"],
    ordered=True,
)

dedup_cols = [c for c in ["track_id", "artists", "track_name", "track_genre"] if c in df.columns]
if dedup_cols:
    df = df.drop_duplicates(subset=dedup_cols, keep="first")

final_columns = [
    "track_name", "artists", "track_genre", "popularity", "duration_ms",
    "danceability", "energy", "valence", "tempo", "loudness", "acousticness",
    "Year", "Popularity_Category",
]
df = df[[c for c in final_columns if c in df.columns]].copy()

print("Cleaned shape:", df.shape)
print("Year range:", int(df["Year"].min()), "to", int(df["Year"].max()))
print("Category counts:", df["Popularity_Category"].value_counts().to_dict())


## 6. Save Cleaned Dataset


In [7]:
data_dir_output = DATA_DIR / "spotify_cleaned.csv"
df.to_csv(data_dir_output, index=False)

print(f"Saved cleaned file: {data_dir_output}")
print("Final columns:", list(df.columns))


Saved cleaned file: /home/ahmed/Downloads/data-visualization-SpotifyTrends-main/spotify_trends_dashboard/dataset_5a/spotify_cleaned.csv
Final columns: ['track_name', 'artists', 'track_genre', 'popularity', 'duration_ms', 'danceability', 'energy', 'valence', 'tempo', 'loudness', 'acousticness', 'Year', 'Popularity_Category']


## Run Dashboard

After running all cells, execute `python app.py` from `spotify_trends_dashboard` or run `./run.sh` from the project root.
